In [16]:
import pandas as pd
from multicall import Call
import numpy as np
from web3 import Web3


from mainnet_launch.database.schema.full import (
    RebalancePlan,
    DestinationTokenValues,
    TokenValues,
    Autopools,
    DestinationStates,
    DestinationTokens,
    Destinations,
    AutopoolDestinationStates,
    Tokens,
    AutopoolStates,
)
import plotly.express as px
import time
import concurrent.futures
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import json


from mainnet_launch.abis import STATS_CALCULATOR_REGISTRY_ABI
from mainnet_launch.data_fetching.get_events import fetch_events


from mainnet_launch.database.schema.postgres_operations import (
    get_full_table_as_orm,
    get_full_table_as_df,
    insert_avoid_conflicts,
    get_subset_not_already_in_column,
    natural_left_right_using_where,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    identity_with_bool_success,
    get_state_by_one_block,
)
from mainnet_launch.constants import ALL_CHAINS, ALL_AUTOPOOLS, ROOT_PRICE_ORACLE, ChainData, AutopoolConstants

from mainnet_launch.pages.autopool_diagnostics.lens_contract import (
    fetch_pools_and_destinations_df,
)

In [ ]:
def convert_rebalance_plan_json_to_rebalance_plan_line(
    rebalance_plan_json_key: str, s3_client, autopool: AutopoolConstants
):
    plan = json.loads(
        s3_client.get_object(
            Bucket=autopool.solver_rebalance_plans_bucket,
            Key=rebalance_plan_json_key,
        )["Body"].read()
    )

    plan["rebalance_plan_json_key"] = rebalance_plan_json_key
    plan["autopool_vault_address"] = autopool.autopool_eth_addr
    return plan


def ensure_rebalance_plan_table_is_current():
    s3_client = boto3.client("s3", config=Config(signature_version=UNSIGNED))

    # autopools = get_full_table_as_orm(Autopools)
    for autopool in ALL_AUTOPOOLS:

        solver_plan_paths_on_remote = [
            r["Key"] for r in s3_client.list_objects_v2(Bucket=autopool.solver_rebalance_plans_bucket).get("Contents")
        ]
        plans_not_already_fetched = get_subset_not_already_in_column(
            RebalancePlan,
            RebalancePlan.file_name,
            solver_plan_paths_on_remote,
            where_clause=RebalancePlan.autopool_vault_address == autopool.autopool_eth_addr,
        )

        tokens = get_full_table_as_orm(Tokens, where_clause=Tokens.chain_id == autopool.chain.chain_id)
        destinations = get_full_table_as_orm(
            Destinations, where_clause=Destinations.chain_id == autopool.chain.chain_id
        )

        a_plan = convert_rebalance_plan_json_to_rebalance_plan_line(plans_not_already_fetched[-1], s3_client, autopool)
        return a_plan, destinations, tokens


plan, destinations, tokens = ensure_rebalance_plan_table_is_current()
plan

2025-04-28 11:38:45,056 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-28 11:38:45,056 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM rebalance_plan
            WHERE rebalance_plan.autopool_vault_address = '0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56'
        
2025-04-28 11:38:45,056 INFO sqlalchemy.engine.Engine [cached since 1431s ago] {}
2025-04-28 11:38:45,241 INFO sqlalchemy.engine.Engine COMMIT
2025-04-28 11:38:45,338 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-28 11:38:45,340 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM tokens
            WHERE tokens.chain_id = 1
        
2025-04-28 11:38:45,341 INFO sqlalchemy.engine.Engine [generated in 0.00096s] {}
2025-04-28 11:38:45,551 INFO sqlalchemy.engine.Engine COMMIT
2025-04-28 11:38:45,641 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-28 11:38:45,642 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM destinations
            WHERE destination

{'timestamp': 1745847039,
 'sodOnly': False,
 'chainId': '1',
 'solverAddress': '0x952D7a7eB2e0804d37d9244BE8e47341356d2f5D',
 'poolAddress': '0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56',
 'destinationOut': '0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56',
 'tokenOut': '0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2',
 'amountOut': '128865845535805784064',
 'amountOutETH': '128865845535805784064',
 'destinationIn': '0x3F55eedDe51504E6Ed0ec30E8289b4Da11EdB7F9',
 'tokenIn': '0xe080027Bd47353b5D1639772b4a75E9Ed3658A0d',
 'minAmountIn': '123375446482083250176',
 'minAmountInETH': '128820552879264989184',
 'steps': [{'stepType': 'swap',
   'dex': '0xV2',
   'tokenOut': '0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2',
   'amountOut': '61304214261994627072',
   'tokenIn': '0xf1C9acDc66974dFB6dEcB12aA385b9cD01190E38',
   'minAmountIn': '0',
   'payload': {'blockNumber': '22367874',
    'buyAmount': '58615327036342083166',
    'buyToken': '0xf1c9acdc66974dfb6decb12aa385b9cd01190e38',
    'fees': {'integra

In [24]:
destinations

[Destinations(destination_vault_address='0xf3ae3c74EaD129e770A864CeE291A805b170bBe0', chain_id=1, exchange_name='balancer', block_deployed=20756405, name='Tokemak-Wrapped Ether-Balancer rETH Stable Pool', symbol='toke-WETH-B-rETH-STABLE', pool_type='balMetaStable', pool='0x1E19CF2D73a72Ef1332C882F20534B6519Be0276', underlying='0x1E19CF2D73a72Ef1332C882F20534B6519Be0276', underlying_symbol='B-rETH-STABLE', underlying_name='Balancer rETH Stable Pool', denominated_in='0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2'),
 Destinations(destination_vault_address='0x865e59D439BF7310c9BC6117E6020B8C87De4065', chain_id=1, exchange_name='balancer', block_deployed=20756406, name='Tokemak-Wrapped Ether-Balancer osETH/wETH StablePool', symbol='toke-WETH-osETH/wETH-BPT', pool_type='balCompStable', pool='0xDACf5Fa19b1f720111609043ac67A9818262850c', underlying='0xDACf5Fa19b1f720111609043ac67A9818262850c', underlying_symbol='osETH/wETH-BPT', underlying_name='Balancer osETH/wETH StablePool', denominated_in='0x

In [ ]:
# Destinations(destination_vault_address='0xD6e262094FFa407068b0BEf6bdBcd16DB982b59E', chain_id=1, exchange_name='aave', block_deployed=22347878, name='Tokemak-USD Coin-Wrapped Aave Ethereum USDT', symbol='toke-USDC-waEthUSDT', pool_type='aTokenStataV3', pool='0x7Bc3485026Ac48b6cf9BaF0A377477Fff5703Af8', underlying='0x7Bc3485026Ac48b6cf9BaF0A377477Fff5703Af8', underlying_symbol='waEthUSDT', underlying_name='Wrapped Aave Ethereum USDT', denominated_in='0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48'),
# might be wrong

In [ ]:
def _rebalance_plan_json_to_rebalance_plan_json(
    plan: dict, autopool: AutopoolConstants, token: list[Tokens], destinations: list[Destinations]
) -> RebalancePlan:

    underlying_out_symbol = [
        d.underlying_symbol
        for d in destinations
        if d.destination_vault_address == Web3.toChecksumAddress(plan["destinationOut"])
    ][0]
    underlying_in_symbol = [
        d.underlying_symbol
        for d in destinations
        if d.destination_vault_address == Web3.toChecksumAddress(plan["destinationIn"])
    ][0]

    token_out_decimals = [t.decimals for t in tokens if t.token_address == Web3.toChecksumAddress(plan["tokenOut"])][0]

    token_in_decimals = [t.decimals for t in tokens if t.token_address == Web3.toChecksumAddress(plan["tokenIn"])][0]

    return RebalancePlan(
        file_name=plan["rebalance_plan_json_key"],
        datetime_generated=pd.to_datetime(int(plan["timestamp"]), unit="s", utc=True),
        autopool_vault_address=plan["autopool_vault_address"],
        chain_id=int(plan["chainId"]),
        dex_aggregator=plan["steps"][0]["dex"],  # not certain here
        solver_address=Web3.toChecksumAddress(plan["solverAddress"]),
        rebalance_type=plan["rebalanceTest"]["type"],
        destination_in=Web3.toChecksumAddress(plan["destinationOut"]),
        token_in=Web3.toChecksumAddress(plan["tokenOut"]),
        destination_out=Web3.toChecksumAddress(plan["destinationIn"]),
        token_out=Web3.toChecksumAddress(plan["tokenIn"]),
        move_name=f"{underlying_out_symbol} -> {underlying_in_symbol}",
        # NOTE: this might amountOutETH might be different for autoUSD, not certain what decimals it is
        amount_out=int(plan["amountOut"]) / (10**token_out_decimals),
        amount_out_safe_value=int(plan["amountOutETH"]) / (10**token_out_decimals),
        min_amount_in=int(plan["minAmountIn"]) / (10**token_in_decimals),
        min_amount_in_safe_value=int(plan["minAmountInETH"]) / (10**token_in_decimals),
        # rebalanceTest values
        out_spot_eth=int(plan["rebalanceTest"]["outSpotETH"]) / (10**token_out_decimals),
        out_dest_apr=float(plan["rebalanceTest"]["outDestApr"]),
        in_spot_eth=int(plan["rebalanceTest"]["inSpotETH"]) / (10**token_in_decimals),
        in_dest_apr=float(plan["rebalanceTest"]["outDestApr"]),
        in_dest_adj_apr=float(plan["rebalanceTest"]["inDestAdjApr"]),
        apr_delta=float(plan["rebalanceTest"]["inDestAdjApr"]) - float(plan["rebalanceTest"]["outDestApr"]),
        swap_offset_period=int(plan["rebalanceTest"]["swapOffsetPeriod"]),
        candidate_destinations=None,  # stub not certain if this should be a list or otherwise
        candidate_destinations_rank=None,  # stub
        #          'rebalanceTest': {'currentTimestamp': 1745847039,
        #   'type': 'FromIdle',
        #   'outDest': 'Autopool',
        #   'outSpotETH': 1.2886584553580578e+20,
        #   'outDestApr': 0,
        #   'inDest': 'Tokemak-Wrapped Ether-osETH/rETH',
        #   'inSpotETH': 1.2882292987566273e+20,
        #   'inDestApr': 0.06753612298754073,
        #   'inDestAdjApr': 0.06439188622228886,
        #   'swapOffsetPeriod': 40},
        # for readability
    )

SyntaxError: invalid syntax (3857098292.py, line 10)

In [27]:
plan

{'timestamp': 1745847039,
 'sodOnly': False,
 'chainId': '1',
 'solverAddress': '0x952D7a7eB2e0804d37d9244BE8e47341356d2f5D',
 'poolAddress': '0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56',
 'destinationOut': '0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56',
 'tokenOut': '0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2',
 'amountOut': '128865845535805784064',
 'amountOutETH': '128865845535805784064',
 'destinationIn': '0x3F55eedDe51504E6Ed0ec30E8289b4Da11EdB7F9',
 'tokenIn': '0xe080027Bd47353b5D1639772b4a75E9Ed3658A0d',
 'minAmountIn': '123375446482083250176',
 'minAmountInETH': '128820552879264989184',
 'steps': [{'stepType': 'swap',
   'dex': '0xV2',
   'tokenOut': '0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2',
   'amountOut': '61304214261994627072',
   'tokenIn': '0xf1C9acDc66974dFB6dEcB12aA385b9cD01190E38',
   'minAmountIn': '0',
   'payload': {'blockNumber': '22367874',
    'buyAmount': '58615327036342083166',
    'buyToken': '0xf1c9acdc66974dfb6decb12aa385b9cd01190e38',
    'fees': {'integra

In [ ]:
# # needed
# class RebalancePlan(Base):
#     __tablename__ = "rebalance_plan"

#     file_name: Mapped[str] = mapped_column(nullable=False, primary_key=True)

#     datetime_generated: Mapped[pd.Timestamp] = mapped_column(DateTime(timezone=True), nullable=False)
#     autopool_vault_address: Mapped[str] = mapped_column(nullable=False)
#     chain_id: Mapped[int] = mapped_column(nullable=False)

#     dex_aggregator: Mapped[str] = mapped_column(nullable=False)

#     solver_address: Mapped[str] = mapped_column(nullable=False)
#     rebalance_type: Mapped[str] = mapped_column(nullable=False)

#     # sometimes this has different destinations but the same underlying token. that means
#     destination_out: Mapped[str] = mapped_column(nullable=False)
#     token_out: Mapped[str] = mapped_column(nullable=False)

#     destination_in: Mapped[str] = mapped_column(nullable=False)
#     token_in: Mapped[str] = mapped_column(nullable=False)

#     move_name: Mapped[str] = mapped_column(nullable=False)  # f"{data['destinationOut']} -> {data['destinationIn']}"

#     amount_out: Mapped[float] = mapped_column(nullable=False)

#     # verify if this is safe or spot values
#     amount_out_safe_value: Mapped[float] = mapped_column(nullable=False)
#     min_amount_in_safe_value: Mapped[float] = mapped_column(nullable=False)
#     min_amount_in: Mapped[float] = mapped_column(nullable=False)

#     out_spot_eth: Mapped[float] = mapped_column(nullable=False)
#     out_dest_apr: Mapped[float] = mapped_column(nullable=False)

#     in_dest_apr: Mapped[float] = mapped_column(nullable=False)
#     int_spot_eth: Mapped[float] = mapped_column(nullable=False)
#     in_dest_adj_apr: Mapped[float] = mapped_column(nullable=False)

#     apr_delta: Mapped[float] = mapped_column(nullable=False)
#     swap_offset_period: Mapped[int] = mapped_column(nullable=False)

#     candidate_destinations: Mapped[list[str]] = mapped_column(ARRAY(String), nullable=False)
#     candidate_destinations_rank: Mapped[int] = mapped_column(nullable=False)

#     projected_swap_cost: Mapped[float] = mapped_column(nullable=False)
#     projected_slippage: Mapped[float] = mapped_column(nullable=False)

#     __table_args__ = (
#         ForeignKeyConstraint(
#             ["destination_in", "chain_id"],
#             ["destinations.destination_vault_address", "destinations.chain_id"],
#         ),
#         ForeignKeyConstraint(
#             ["destination_out", "chain_id"],
#             ["destinations.destination_vault_address", "destinations.chain_id"],
#         ),
#         ForeignKeyConstraint(["token_in", "chain_id"], ["tokens.token_address", "tokens.chain_id"]),
#         ForeignKeyConstraint(["token_out", "chain_id"], ["tokens.token_address", "tokens.chain_id"]),
#         ForeignKeyConstraint(
#             ["autopool_vault_address", "chain_id"], ["autopools.autopool_vault_address", "autopools.chain_id"]
#         ),
#     )

#     # dex steps here?